# Trapped-Ion QCCD Framework — Getting Started

End-to-end demo of the **new** `trapped_ion/` module for QCCD hardware simulation.

**Pipeline**:
```
RotatedSurfaceCode → CSSMemoryExperiment → ideal stim.Circuit
    → TrappedIonCompiler.compile(ideal)
        → decompose_to_native → map_qubits → route → schedule
    → CompiledCircuit  (.scheduled.batches, .mapping, .metrics)
    → display_architecture()  — static figure
    → animate_transport()     — step-by-step animation
    → TrappedIonNoiseModel.apply_with_plan()  — noisy circuit
    → PyMatchingDecoder  — logical error rate
```

Covers both **Augmented Grid** and **WISE** architectures on a **d=2 rotated surface code**.

In [ ]:
"""Imports."""
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

# -- Trapped-ion module (new API) --
from qectostim.experiments.hardware_simulation.trapped_ion.utils import (
    AugmentedGridArchitecture,
    WISEArchitecture,
    TrappedIonCompiler,
    TrappedIonExperiment,
    TrappedIonNoiseModel,
)
from qectostim.experiments.hardware_simulation.trapped_ion.utils.qccd_nodes import (
    QCCDWiseArch,
)
from qectostim.experiments.hardware_simulation.trapped_ion.viz import (
    display_architecture,
    animate_transport,
    render_animation,
)
from qectostim.experiments.hardware_simulation.trapped_ion.demo.run import (
    build_viz_mappings,
)

# -- Circuit generation --
from qectostim.codes.surface.rotated_surface import RotatedSurfaceCode
from qectostim.experiments.memory import CSSMemoryExperiment

# -- Visualization --
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import numpy as np

print('All imports OK')

## Build Ideal Circuit

Generate a d=2 rotated surface code memory experiment (1 round, Z-basis).

In [ ]:
"""Build ideal stim circuit."""
code = RotatedSurfaceCode(distance=2)
mem = CSSMemoryExperiment(code=code, rounds=2, noise_model=None, basis='z')
ideal = mem.to_stim()

print(f'Code:    d={code.distance}, n={code.n}')
print(f'Circuit: {len(ideal)} instructions, {ideal.num_qubits} qubits')
print()
print(ideal)

---
## Augmented Grid

Compile, route, parallelise, and visualise on an Augmented Grid architecture.

In [ ]:
"""Augmented Grid — compile."""
arch_ag = AugmentedGridArchitecture(
    trap_capacity=2, rows=3, cols=3, padding=0,
)
compiler_ag = TrappedIonCompiler(arch_ag, is_wise=False)
compiled_ag = compiler_ag.compile(ideal)

# Visualization mappings
ion_roles_ag, p2l_ag, remap_ag = build_viz_mappings(compiler_ag)

# Metrics
batches_ag = compiled_ag.scheduled.batches
all_ops_ag = compiled_ag.scheduled.metadata.get('all_operations', [])
metrics_ag = compiled_ag.compute_metrics()

print(f'Traps:          {len(arch_ag._manipulationTraps)}')
print(f'Routed ops:     {len(all_ops_ag)}')
print(f'Parallel steps: {len(batches_ag)}')
print(f'Depth:          {metrics_ag.get("depth", "?")}')
print(f'Duration (µs):  {metrics_ag.get("duration_us", "?")}')
print(f'Ion roles:      {ion_roles_ag}')
if batches_ag:
    print(f'First batch:    {batches_ag[0].label}')

In [ ]:
"""Augmented Grid — static architecture layout."""
arch_ag.resetArrangement()
arch_ag.refreshGraph()

fig, ax = display_architecture(
    arch_ag,
    title='Augmented Grid — initial layout',
    show_junctions=True,
    show_edges=True,
    show_ions=True,
    show_labels=False,
    show_legend=True,
    figsize=(14, 10),
    ion_roles=ion_roles_ag,
    ion_idx_remap=remap_ag,
    physical_to_logical=p2l_ag,
)
fig.tight_layout()
plt.show()

In [ ]:
"""Augmented Grid — animate transport."""
arch_ag.resetArrangement()
arch_ag.refreshGraph()

ani_ag = animate_transport(
    arch=arch_ag,
    operations=batches_ag,
    interval=600,
    show_labels=True,
    interp_frames=2,
    gate_hold_frames=1,
    stim_circuit=ideal,
    figsize=(16, 10),
    title_prefix='Augmented Grid',
    repeat=True,
    ion_roles=ion_roles_ag,
    ion_idx_remap=remap_ag,
    physical_to_logical=p2l_ag,
)

# Inline display (jshtml fallback — works without ffmpeg)
display(HTML(ani_ag.to_jshtml()))

In [ ]:
"""Augmented Grid — render to MP4 (requires ffmpeg)."""
# Reset again (animate_transport is destructive)
arch_ag.resetArrangement()
arch_ag.refreshGraph()

ani_ag_mp4 = animate_transport(
    arch=arch_ag,
    operations=batches_ag,
    interval=600,
    show_labels=True,
    interp_frames=2,
    gate_hold_frames=1,
    stim_circuit=ideal,
    figsize=(16, 10),
    title_prefix='Augmented Grid',
    repeat=False,
    ion_roles=ion_roles_ag,
    ion_idx_remap=remap_ag,
    physical_to_logical=p2l_ag,
)

render_animation(
    ani_ag_mp4,
    output_path='aug_grid_transport.mp4',
    fps=5,
    dpi=120,
    display_inline=True,
    fallback_to_jshtml=True,
)

---
## WISE Architecture

Compile, route, parallelise, and visualise on a WISE (Wired Ion Surface Electrode) architecture.

In [ ]:
"""WISE — compile with full routing control."""
from qectostim.experiments.hardware_simulation.trapped_ion.compiler.routing_config import (
    WISERoutingConfig,
)

# ── Architecture + compiler ───────────────────────────────────────
wise_cfg = QCCDWiseArch(m=1, n=3, k=4)
arch_w = WISEArchitecture(
    wise_config=wise_cfg,
    add_spectators=True,
    compact_clustering=True,
)
compiler_w = TrappedIonCompiler(
    arch_w, is_wise=True, wise_config=wise_cfg,
)

# Production defaults: all CPU cores, base_pmax_in=1, tqdm progress.
# Override any parameter as needed, e.g. lookahead=4, subgridsize=(6,6,1).
compiler_w.routing_kwargs = dict(
    routing_config=WISERoutingConfig.default(
        lookahead=4,
        subgridsize=(6, 6, 1),
    ),
)

compiled_w = compiler_w.compile(ideal)

# Visualization mappings
ion_roles_w, p2l_w, remap_w = build_viz_mappings(compiler_w)

# Metrics
batches_w = compiled_w.scheduled.batches
all_ops_w = compiled_w.scheduled.metadata.get('all_operations', [])
metrics_w = compiled_w.compute_metrics()

print(f'Traps:          {len(arch_w._manipulationTraps)}')
print(f'Routed ops:     {len(all_ops_w)}')
print(f'Parallel steps: {len(batches_w)}')
print(f'Depth:          {metrics_w.get("depth", "?")}')
print(f'Duration (µs):  {metrics_w.get("duration_us", "?")}')
print(f'Ion roles:      {ion_roles_w}')
if batches_w:
    print(f'First batch:    {batches_w[0].label}')

In [ ]:
"""WISE — static architecture layout."""
arch_w.resetArrangement()
arch_w.refreshGraph()

display_architecture(
    arch_w,
    title='WISE — initial layout',
    show_junctions=True,
    show_edges=True,
    show_ions=True,
    show_labels=True,
    show_legend=True,
    figsize=(16, 12),
    ion_roles=ion_roles_w,
    ion_idx_remap=remap_w,
    physical_to_logical=p2l_w,
)
plt.tight_layout()
plt.show()

In [ ]:
"""WISE — animate transport."""
arch_w.resetArrangement()
arch_w.refreshGraph()

ani_w = animate_transport(
    arch=arch_w,
    operations=batches_w,
    interval=600,
    show_labels=True,
    interp_frames=2,
    gate_hold_frames=1,
    stim_circuit=ideal,
    figsize=(16, 12),
    title_prefix='WISE',
    repeat=True,
    ion_roles=ion_roles_w,
    ion_idx_remap=remap_w,
    physical_to_logical=p2l_w,
)

display(HTML(ani_w.to_jshtml()))

In [ ]:
"""WISE — render to MP4 (requires ffmpeg)."""
arch_w.resetArrangement()
arch_w.refreshGraph()

ani_w_mp4 = animate_transport(
    arch=arch_w,
    operations=batches_w,
    interval=600,
    show_labels=True,
    interp_frames=2,
    gate_hold_frames=1,
    stim_circuit=ideal,
    figsize=(16, 12),
    title_prefix='WISE',
    repeat=False,
    ion_roles=ion_roles_w,
    ion_idx_remap=remap_w,
    physical_to_logical=p2l_w,
)

render_animation(
    ani_w_mp4,
    output_path='wise_transport.mp4',
    fps=5,
    dpi=120,
    display_inline=True,
    fallback_to_jshtml=True,
)

---
## Noisy Simulation

Apply hardware noise (dephasing, gate errors, transport errors) and decode
with PyMatching to measure the logical error rate.

In [ ]:
"""Noisy simulation — Augmented Grid."""
from qectostim.decoders.pymatching_decoder import PyMatchingDecoder

# Fresh architecture for simulation
arch_sim = AugmentedGridArchitecture(
    trap_capacity=2, rows=3, cols=3, padding=0,
)
compiler_sim = TrappedIonCompiler(arch_sim, is_wise=False)
noise = TrappedIonNoiseModel()

experiment = TrappedIonExperiment(
    code=code,
    architecture=arch_sim,
    compiler=compiler_sim,
    hardware_noise=noise,
    rounds=1,
    basis='z',
)

# Build → compile → noise
ideal_circ = experiment.build_ideal_circuit()
experiment._compiled = experiment.compiler.compile(ideal_circ)
noisy_circuit = experiment.apply_hardware_noise()

print(f'Noisy circuit: {len(noisy_circuit)} instructions')
print(f'Has noise:     {any("ERROR" in str(inst) for inst in noisy_circuit)}')
print()

# Decode
num_shots = 1_000
dem = noisy_circuit.detector_error_model(
    decompose_errors=True,
    approximate_disjoint_errors=True,
)
decoder = PyMatchingDecoder(dem=dem)

sampler = noisy_circuit.compile_detector_sampler()
detection_events, observable_flips = sampler.sample(
    num_shots, separate_observables=True,
)
predictions = decoder.decode_batch(detection_events)
num_errors = int(np.sum(np.any(predictions != observable_flips, axis=1)))
ler = num_errors / num_shots

print(f'Shots:              {num_shots}')
print(f'Logical errors:     {num_errors}')
print(f'Logical error rate: {ler:.4f}')

---
## Pipeline Reference

| Stage | Class / Function | Description |
|-------|-----------------|-------------|
| **Code** | `RotatedSurfaceCode(d)` | QEC code definition |
| **Ideal circuit** | `CSSMemoryExperiment → .to_stim()` | Stabiliser measurement circuit |
| **Architecture** | `AugmentedGridArchitecture` / `WISEArchitecture` | QCCD topology |
| **Compiler** | `TrappedIonCompiler.compile(circuit)` | Full pipeline |
| &nbsp;&nbsp;Decompose | `.decompose_to_native()` | stim → MS + R{X,Y} + M + R |
| &nbsp;&nbsp;Map | `.map_qubits()` | Logical → physical ions (clustering) |
| &nbsp;&nbsp;Route | `.route()` | Ion shuttling (WISE SAT / heuristic) |
| &nbsp;&nbsp;Schedule | `.schedule()` | Parallel batches + timing |
| **Display** | `display_architecture()` | Static matplotlib figure |
| **Animate** | `animate_transport()` | FuncAnimation with interpolation |
| **Render** | `render_animation()` | MP4 via ffmpeg (or jshtml fallback) |
| **Noise** | `TrappedIonNoiseModel.apply_with_plan()` | Dephasing + gate + transport errors |
| **Decode** | `PyMatchingDecoder.decode_batch()` | MWPM decoding |
| **Experiment** | `TrappedIonExperiment.simulate()` | Full simulation loop |